## From theory to practice

From Chapter 2 you should have a good grasp of NumPy, JAX, and PyTorch’s torch.tensor. This is all that is needed for this chapter, and nothing else is required. From the next chapter we will progress to their higher-level APIs.

I suggest a short exercise to let you train your first differentiable model from scratch:

1. Load a toy dataset: for example, one of those contained in scikit-learn datasets module.
2. Build a linear model (for regression or classification depending on the dataset). Think about how to make the code as modular as possible: as we will see, you will need at least two functions, one for initializing the parameters of the model and one for computing the model’s predictions.
4. Train the model via gradient descent. For now you can compute the gradients manually: try to imagine how you can make also this part modular, i.e., how do you change the gradient’s computation if you want to dynamically add or remove the bias from a model?
5. Plot the loss function and the accuracy on an independent test set. If you know some standard
machine learning, you can compare the results to other supervised learning models, such as a decision tree or a k-NN, always using scikit-learn.

In [1]:
!uv pip install matplotlib scikit-learn --quiet
!uv pip install torch --quiet

## 1. Load dataset

In [2]:
from sklearn.datasets import load_iris
import torch

In [3]:
data = load_iris()

In [4]:
X = torch.from_numpy(data.data).float()
y = torch.nn.functional.one_hot(torch.from_numpy(data.target)).float()

In [5]:
n = X.shape[0]  # size of the dataset
c = X.shape[1]  # features
m = y.shape[1]  # classes

print(f"Size of dataset n = {n}")
print(f"Number of features c = {c}")
print(f"Classes m = {m}")

Size of dataset n = 150
Number of features c = 4
Classes m = 3


In [6]:
from sklearn.model_selection import train_test_split

# Split dataset 80% train / 20% test for step 4
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=data.target
)

## 2. Logistic regression model

In [7]:
class LogisticRegression:

    def __init__(self, m, c):
        self.W = torch.normal(0, 1/c**0.5, size=(m, c), requires_grad=True)
        self.b = torch.zeros(m, requires_grad=True)

    @staticmethod
    def softmax(logits):
        # Note: very unstable
        # sumexp = torch.sum(torch.exp(logits), dim=-1, keepdim=True)
        # return torch.exp(logits) / sumexp
        # Somewhat more stable but worse than logsoftmax
        exp = torch.exp(logits - torch.amax(logits, -1, keepdim=True))
        return exp / torch.sum(exp, dim=-1, keepdim=True)

    def forward(self, X, logits=False):
        if logits:
            return X @ self.W.T + self.b
        return self.softmax(X @ self.W.T + self.b)

In [8]:
def logsoftmax(logits):
    logits_shifted = logits - torch.amax(logits, -1, keepdim=True)
    return logits_shifted - torch.log(torch.sum(torch.exp(logits_shifted), dim=-1, keepdim=True))

In [9]:
def cross_entropy_loss(y, y_pred, from_logits=False):
    if from_logits:
        return -torch.mean(y * logsoftmax(y_pred))
    return -torch.mean(y * torch.log(y_pred))

## 3. Train

The book doesn't derive the gradient for the logistic regression. The softmax function makes it a bit complicated. So we use torch's autograd for gradient descent.

In [65]:
learning_rate = 1e-2
momentum = 0.9
epochs = 500

In [66]:
lr_model = LogisticRegression(m, c)
gradW = torch.zeros_like(lr_model.W)
gradb = torch.zeros_like(lr_model.b)

In [67]:
for epoch in range(epochs):
    # Let the autograd compute model gradients
    lr_model.W.grad = None
    lr_model.b.grad = None
    # Predict
    y_pred = lr_model.forward(X_train, logits=False)
    loss = cross_entropy_loss(y_train, y_pred, from_logits=False)
    loss.backward()
    with torch.no_grad():
        gradW = -learning_rate * lr_model.W.grad + momentum * gradW
        lr_model.W += gradW
        gradb = -learning_rate * lr_model.b.grad + momentum * gradb
        lr_model.b += gradb
    # print(f"Epoch {epoch} - loss: {loss}")

## 4. Validation

We'll do the same loop but save test accuracy.

In [68]:
pred = lr_model.forward(X_test).argmax(dim=1)

In [69]:
(pred == y_test.argmax(dim=1)).float().mean()

tensor(1.)